# 01 - Exploratory Data Analysis

**Model objective:** classify task duration into duration buckets, such as Short, Medium, Long, or Story Task, using relevant raw Jira ticket fields.

#### 01-01 Dataset Shape Inspection

First, identify the dataset shape to confirm the number of rows and columns available for analysis.



In [1]:
import pandas as pd

ticket_df = pd.read_csv('../jira_ticket_dataset.csv')
pd.set_option('display.max_rows', None)

ticket_dataset_shape = ticket_df.shape
print(ticket_dataset_shape)

C:\Users\Omar\AppData\Local\Temp\ipykernel_39100\2846052727.py:3: DtypeWarning: Columns (0: customfield_12310921, 1: issuetype.subtask) have mixed types. Specify dtype option on import or set low_memory=False.
  ticket_df = pd.read_csv('../jira_ticket_dataset.csv')


(1149323, 37)


Next, review the available columns, non-null counts, and inferred data types.

In [2]:
ticket_df.head().info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id                           5 non-null      float64
 1   key                          5 non-null      str    
 2   summary                      5 non-null      str    
 3   resolution.id                3 non-null      float64
 4   resolution.description       3 non-null      str    
 5   resolution.name              3 non-null      str    
 6   priority.id                  5 non-null      float64
 7   priority.name                5 non-null      str    
 8   labels                       5 non-null      str    
 9   assignee                     2 non-null      str    
 10  status.id                    5 non-null      float64
 11  status.description           5 non-null      str    
 12  status.name                  5 non-null      str    
 13  statusCategory.id            5 non-

#### 01-02 Feature Extraction

Based on the dataframe inspection, the most relevant raw features to keep for initial modeling are:

- **summary**: ticket title or short task description

- **description**: longer issue details

- **priority.name**: priority category

- **labels**: assigned Jira labels

- **subtasks**: linked subtasks

- **issuetype.description**: description of the issue type

- **issuetype.name**: issue type label

- **issuetype.subtask**: whether the issue is a subtask

- **resolutiondate**: timestamp used to derive duration

- **created**: timestamp used to derive duration

- **watches.watchCount**: number of watchers


Some raw features will later be transformed into engineered model inputs:


- **summary**: can be used for text length and keyword features

- **description**: can be used for text length and keyword features

- **labels**: can be used for label-count features

- **resolutiondate** and **created**: can be used to derive task duration


A crucial target-related feature is **duration_days**, derived from the difference between the resolution timestamp and the creation timestamp.


```python

duration_days = resolutiondate - created

```


The planned duration buckets are:


- 0-3 days: Short

- 4-7 days: Medium

- 8-14 days: Long

- 15+ days: Story Task



In [3]:
raw_feature_columns = [

    "summary",
    "description",
    "priority.name",
    "labels",
    "votes.votes",
    "issuetype.description",
    "issuetype.name",
    "issuetype.subtask",
    "resolutiondate",
    "created",
    "watches.watchCount"

]

raw_feature_sample = ticket_df.loc[:, raw_feature_columns].sample(n=10,random_state=42)
print(raw_feature_sample)

                                                  summary  \
362921  ensure DONE status is set when cxf bc consumer...   
431071  Review UIMA-AS support for running with embedd...   
580054  Set classpath in MiniCluster only when it is n...   
164680          Remove the executor count from the header   
994340    ListingBasedRollbackStrategy support logcompact   
745408              Remove method register in Interpreter   
703214  Translate the "Distributed Runtime Environment...   
165951  Some renderers were not calling their "toStrin...   
711160      Streaming File Sink s3 end-to-end test stalls   
141541  Add missing rules to standard rules page and r...   

                                              description priority.name  \
362921             this can prevent potential thread leak         Major   
431071  UIMA-AS needs to support embedded AMQ broker. ...         Major   
580054                                                NaN         Major   
164680                      